## 0. Setup shared by the three validation notebooks
Paths are relative to the repository root. The statistics come from `tools/validation_metrics.py`
(Ministry of Transport guideline, draft 6, September 2024); results go to `Output/validation_mot/`.

In [1]:
import os, sys, warnings
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'
sys.path.insert(0, 'tools')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import validation_metrics as vm
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6'
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/validation_mot'; FIG = 'Output/figures'
# Time window (wave 1b): AM 06:00-09:00 is the base and the default; PM 16:00-19:00 and midday 10:00-14:00 with the environment variable NOFIT_PERIOD
PERIOD = os.environ.get('NOFIT_PERIOD', 'AM')
HOURS, SUF = {'AM': ([6, 7, 8], ''), 'PM': ([16, 17, 18], '_pm'), 'MD': ([10, 11, 12, 13], '_md')}[PERIOD]
PLABEL = {'AM': 'AM 06:00-09:00', 'PM': 'PM 16:00-19:00', 'MD': 'midday 10:00-14:00'}[PERIOD]
W = {'AM': '06-09', 'PM': '16-19', 'MD': '10-14'}[PERIOD]
PSUF = '' if PERIOD == 'AM' else '_' + PERIOD            # suffix of the validation output files of the other periods
def vfile(name): return f'{OUT}/{name}{PSUF}.csv'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
def need(*paths):
    for p in paths: assert not is_pointer(p), f'{p} is a Git LFS pointer: python3 tools/lfs_pull.py {p}'

SUMMARY = []      # one row per guideline check; written to Output/validation_mot/summary_<stage>.csv
def add_summary(test, ref, item, level, criterion, result, status, explanation, data):
    SUMMARY.append({'period': PERIOD, 'test': test, 'guideline': ref, 'item': item, 'level': level, 'criterion': criterion,
                    'result': result, 'status': status, 'explanation': explanation, 'data (vintage)': data})

def scatter_origin(ax, obs, mod, title, xlabel, ylabel, logscale=False, color=BLUE):
    """Guideline format: observed on x, model on y, regression through the origin with equation and R2."""
    o = np.asarray(obs, float); m = np.asarray(mod, float); ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    ax.scatter(o, m, s=10, color=color, alpha=0.55, linewidths=0)
    hi = max(o.max(), m.max()) * 1.05 if len(o) else 1
    ax.plot([0, hi], [0, hi], color=MUTED, lw=0.8, ls=':')
    b = vm.slope_origin(o, m); r2 = vm.r2_correl(o, m)
    ax.plot([0, hi], [0, b * hi], color=ORANGE, lw=1.5)
    ax.text(0.04, 0.95, f'y = {b:.2f}x   R² = {r2:.2f}   n = {len(o)}', transform=ax.transAxes, va='top', fontsize=9, color=INK)
    if logscale: ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    style_ax(ax, title, xlabel, ylabel)

def save_fig(fig, name):
    fig.savefig(f'{FIG}/{name.replace(".png", PSUF + ".png")}', dpi=150, bbox_inches='tight', facecolor='white'); plt.close(fig)

# Bus coverage by period and by trip purpose (wave 1b finding)

**Question.** RavKav records 0.74 of the survey's AM bus trips (study-wide), but 0.96 in the PM peak and 0.93 at midday (steps 15 and 16 run for the three windows). If the shortfall were a property of the
ticketing extract (operators, cash fares, un-geocoded stops) it should not depend on the hour. What differs between the windows is the mix of trips: in the survey, home-based education trips
are a much larger share of the bus trips in the morning. This notebook shows the ratio by window, with and without the education trips, from the survey's own purpose coding (home-based work, education, other; non-home-based).
It is a diagnostic of the coverage rule, not a change to it.

In [2]:
TRIPS = 'Input/THS_2017-2018/trips_ths_2017.xlsx'; KEYS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
need(TRIPS, KEYS, 'Input/TAZ_2636_Keys.xlsx')
keys_raw = pd.read_csv(KEYS, encoding='windows-1255')
Z1250 = set(keys_raw['TAZ_1270'].dropna().astype(int).unique()) if False else set(keys_raw.dropna(subset=['TAZ_NUMBER'])['TAZ_1270'].unique())
df = pd.read_excel(TRIPS); d = df.sort_values(['PerID3', 'SurveyDay', 'placeno']).copy(); g = d.groupby(['PerID3', 'SurveyDay'])
d['origin'] = g['actTaz'].shift(1); d['dep_h'] = g['Dep_h'].shift(1); d['o_act'] = g['mainActivity'].shift(1)
t = d.dropna(subset=['origin', 'actTaz', 'dep_h']).copy()
k26 = pd.read_excel('Input/TAZ_2636_Keys.xlsx').drop_duplicates('TAZ_2636').set_index('TAZ_2636')['TAZ_1250']
t['o1250'] = t['origin'].map(k26); t['d1250'] = t['actTaz'].map(k26)
t = t[t.o1250.isin(Z1250) & t.d1250.isin(Z1250) & t.mainmode.isin([3, 5])].copy()          # bus (3 public bus, 5 Metronit), both ends in the study area
def purpose(r):
    o, dd = r['o_act'], r['mainActivity']
    if o == 1 and dd == 1: return 'home-based other'
    if o != 1 and dd != 1: return 'non-home-based'
    a = dd if o == 1 else o
    return 'home-based work' if a in (2, 3) else 'home-based education' if a == 4 else 'home-based other'
t['purpose'] = t.apply(purpose, axis=1); t['w'] = t['new_wf'] / 2
RK = {'AM 06:00-09:00': ('Output/bus/periods/bus_od_taz_avg_AM.csv', [6, 7, 8]), 'PM 16:00-19:00': ('Output/bus/periods/bus_od_taz_avg_PM.csv', [16, 17, 18]), 'midday 10:00-14:00': ('Output/bus/periods/bus_od_taz_avg_MD.csv', [10, 11, 12, 13])}
study = [int(c) for c in pd.read_csv('Output/ths2017/two_mode/car_taz.csv', index_col=0).columns]
rows = []
for wname, (path, hrs) in RK.items():
    m = pd.read_csv(path, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int)
    rk_total = float(m.reindex(index=study, columns=study).fillna(0).values.sum())
    x = t[t.dep_h.isin(hrs)]; tot = x.w.sum(); sh = x.groupby('purpose')['w'].sum() / tot
    ed = x.loc[x.purpose == 'home-based education', 'w'].sum()
    rows.append({'window': wname, 'survey bus trips (expanded, per day)': tot, 'RavKav journeys (per Tuesday)': rk_total, 'RavKav / survey': rk_total / tot,
                 'home-based education share of survey bus trips': ed / tot, 'home-based work share': sh.get('home-based work', 0), 'home-based other share': sh.get('home-based other', 0), 'non-home-based share': sh.get('non-home-based', 0),
                 'RavKav / survey excluding education': rk_total / (tot - ed)})
cov = pd.DataFrame(rows); cov.to_csv(f'{OUT}/BusCoverage_by_period_and_purpose.csv', index=False, float_format='%.4f'); print(cov.round(3).T.to_string())
fig, ax = plt.subplots(figsize=(7.5, 4), facecolor='white'); x = np.arange(len(cov))
ax.bar(x - 0.2, cov['RavKav / survey'], 0.4, color=BLUE, label='RavKav / survey, all trips'); ax.bar(x + 0.2, cov['RavKav / survey excluding education'], 0.4, color=ORANGE, label='RavKav / survey, excluding education trips')
ax.axhline(1, color=MUTED, ls='--', lw=1); ax.set_xticks(x); ax.set_xticklabels(cov['window'], fontsize=9); ax.legend(frameon=False, fontsize=8); style_ax(ax, 'RavKav coverage of the survey\'s bus trips by window', None, 'ratio')
save_fig(fig, 'mot_BusCoverage_by_period.png')

                                                             0               1                   2
window                                          AM 06:00-09:00  PM 16:00-19:00  midday 10:00-14:00
survey bus trips (expanded, per day)                125769.787       82079.119          129834.895
RavKav journeys (per Tuesday)                         92439.75        78490.25           119429.75
RavKav / survey                                          0.735           0.956                0.92
home-based education share of survey bus trips           0.398           0.131               0.189
home-based work share                                    0.314             0.2               0.133
home-based other share                                   0.169           0.408               0.411
non-home-based share                                      0.12           0.261               0.268
RavKav / survey excluding education                       1.22           1.101               1.134


In [3]:
add_summary('T14', '5.2', 'RavKav / survey bus ratio by window, with and without education trips', 'study area, three windows', 'none (diagnostic)',
            '; '.join(f'{r["window"]}: {r["RavKav / survey"]:.2f} (education {r["home-based education share of survey bus trips"]:.0%} of survey trips; {r["RavKav / survey excluding education"]:.2f} without them)' for _, r in cov.iterrows()),
            'finding', 'The AM shortfall of RavKav against the survey (0.74) is absent at midday and in the PM peak (0.93-0.96), when home-based education is 13-19 % of the survey bus trips instead of 40 %. Hypothesis: student / school trips are under-recorded in the extract or over-weighted in the survey; to be checked with the data provider.', 'survey 2017/18; RavKav May 2022')

## Summary of this notebook against the guideline

In [4]:
SUM = pd.DataFrame(SUMMARY); SUM.to_csv(f'{OUT}/summary_stage1c.csv', index=False)
SUM[['test', 'guideline', 'item', 'level', 'criterion', 'result', 'status']]

,test,guideline,item,level,criterion,result,status
0,T14,5.2,"RavKav / survey bus ratio by window, with and ...","study area, three windows",none (diagnostic),AM 06:00-09:00: 0.73 (education 40% of survey ...,finding
